In [1]:
from google.colab import files
uploaded = files.upload()

Saving master_dashboard.csv to master_dashboard.csv
Saving state_month_dashboard.csv to state_month_dashboard.csv
Saving state_year_rates.csv to state_year_rates.csv
Saving state_year_rates_recent_2024_2025.csv to state_year_rates_recent_2024_2025.csv


In [7]:
BASE = "/content"

In [3]:
import os

for f in os.listdir(BASE):
    print(f)


.config
state_month_dashboard.csv
master_dashboard.csv
state_year_rates_recent_2024_2025.csv
state_year_rates.csv
sample_data


In [4]:
import pandas as pd
import numpy as np
import os

master = pd.read_csv(f"{BASE}/master_dashboard.csv")
state_month = pd.read_csv(f"{BASE}/state_month_dashboard.csv")
state_year = pd.read_csv(f"{BASE}/state_year_rates.csv")
state_year_recent = pd.read_csv(f"{BASE}/state_year_rates_recent_2024_2025.csv")

master["month_start"] = pd.to_datetime(master["month_start"])
state_month["month_start"] = pd.to_datetime(state_month["month_start"])

recent = master[master["analysis_window"].eq("Recent: 2024-Jan 2026")].copy()
recent_full = master[master["full_recent_year"].eq("2024-2025 full years")].copy()

print("MASTER:", master.shape)
print("STATE MONTH:", state_month.shape)
print("STATE YEAR:", state_year.shape)
print("STATE YEAR RECENT:", state_year_recent.shape)
print("RECENT:", recent.shape)
print("RECENT FULL YEARS:", recent_full.shape)


MASTER: (58284, 53)
STATE MONTH: (3367, 24)
STATE YEAR: (304, 9)
STATE YEAR RECENT: (16, 9)
RECENT: (2714, 53)
RECENT FULL YEARS: (2608, 53)


In [8]:
# ============================================================
# DVN AT3 - Detailed EDA for Tableau Story
# ============================================================

import pandas as pd
import numpy as np


master = pd.read_csv(f"{BASE}/master_dashboard.csv")
state_month = pd.read_csv(f"{BASE}/state_month_dashboard.csv")
state_year = pd.read_csv(f"{BASE}/state_year_rates.csv")
state_year_recent = pd.read_csv(f"{BASE}/state_year_rates_recent_2024_2025.csv")

master["month_start"] = pd.to_datetime(master["month_start"])
state_month["month_start"] = pd.to_datetime(state_month["month_start"])

recent = master[master["analysis_window"].eq("Recent: 2024-Jan 2026")].copy()
recent_full = master[master["full_recent_year"].eq("2024-2025 full years")].copy()

print("MASTER:", master.shape)
print("STATE MONTH:", state_month.shape)
print("STATE YEAR:", state_year.shape)
print("RECENT FULL:", recent_full.shape)

# 1. Long view: annual fatalities
annual = (
    master.groupby("year", as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

annual["year_type"] = np.where(annual["year"].eq(2026), "Partial year", "Full year")
annual["yoy_change"] = annual["fatalities"].diff()
annual["yoy_pct_change"] = annual["fatalities"].pct_change() * 100

print("\nAnnual trend latest years:")
display(annual.tail(15))

print("\nRecent change 2024 to 2025:")
f2024 = annual.loc[annual["year"].eq(2024), "fatalities"].iloc[0]
f2025 = annual.loc[annual["year"].eq(2025), "fatalities"].iloc[0]
print("2024:", f2024)
print("2025:", f2025)
print("Change:", f2025 - f2024)
print("Percent change:", round((f2025 - f2024) / f2024 * 100, 2), "%")

# 2. COVID caution / baseline comparison
periods = pd.DataFrame({
    "period": ["Pre-COVID baseline", "COVID period", "Recent full years"],
    "years": ["2017-2019", "2020-2021", "2024-2025"],
    "avg_annual_fatalities": [
        annual[annual["year"].between(2017, 2019)]["fatalities"].mean(),
        annual[annual["year"].between(2020, 2021)]["fatalities"].mean(),
        annual[annual["year"].between(2024, 2025)]["fatalities"].mean()
    ]
})
display(periods)

# 3. Raw state counts vs population-adjusted risk
state_raw = (
    recent_full.groupby("state", as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

state_risk = (
    state_year_recent.groupby("state", as_index=False)
    .agg(
        fatalities=("fatalities", "sum"),
        avg_population=("population", "mean")
    )
)
state_risk["fatalities_per_100k"] = state_risk["fatalities"] / state_risk["avg_population"] * 100000
state_risk = state_risk.sort_values("fatalities_per_100k", ascending=False)

print("\nRaw state ranking:")
display(state_raw)

print("\nPopulation-adjusted state ranking:")
display(state_risk)

# 4. Broad column scan for strongest story segments
story_dims = [
    "state", "road_user", "road_user_group", "age_band", "gender",
    "remoteness_group", "speed_band", "national_road_type",
    "crash_type", "time_band", "dayweek", "holiday_period",
    "heavy_vehicle_involved", "rain_context", "heat_context", "wind_context"
]

for col in story_dims:
    if col in recent.columns:
        out = (
            recent.groupby(col, dropna=False)
            .agg(fatalities=("deaths", "sum"))
            .reset_index()
            .sort_values("fatalities", ascending=False)
        )
        out["share_pct"] = out["fatalities"] / out["fatalities"].sum() * 100
        print(f"\n=== {col.upper()} ===")
        display(out.head(12))

# 5. Pattern finder: where risk concentrates
risk_segments = (
    recent.groupby(["state", "remoteness_group", "speed_band", "road_user_group"], dropna=False)
    .agg(fatalities=("deaths", "sum"))
    .reset_index()
    .sort_values("fatalities", ascending=False)
)
risk_segments["share_pct"] = risk_segments["fatalities"] / risk_segments["fatalities"].sum() * 100

print("\nTop risk segments:")
display(risk_segments.head(25))

# 6. Timing pattern
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
time_order = ["Late night 00-05", "Morning commute 06-08", "Daytime 09-15", "Evening commute 16-18", "Night 19-23", "Unknown"]

day_time = (
    recent.groupby(["dayweek", "time_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

day_time["dayweek"] = pd.Categorical(day_time["dayweek"], categories=day_order, ordered=True)
day_time["time_band"] = pd.Categorical(day_time["time_band"], categories=time_order, ordered=True)
day_time = day_time.sort_values(["dayweek", "time_band"])

print("\nDay x time heatmap data:")
display(day_time)

# 7. Export EDA outputs for Tableau / documentation
OUT = "/content/eda_outputs"
import os
os.makedirs(OUT, exist_ok=True)

annual.to_csv(f"{OUT}/eda_annual_trend.csv", index=False)
periods.to_csv(f"{OUT}/eda_period_comparison.csv", index=False)
state_raw.to_csv(f"{OUT}/eda_state_raw_recent_2024_2025.csv", index=False)
state_risk.to_csv(f"{OUT}/eda_state_population_adjusted_recent_2024_2025.csv", index=False)
risk_segments.to_csv(f"{OUT}/eda_top_risk_segments_recent.csv", index=False)
day_time.to_csv(f"{OUT}/eda_day_time_recent.csv", index=False)

print("EDA outputs saved to:", OUT)


MASTER: (58284, 53)
STATE MONTH: (3367, 24)
STATE YEAR: (304, 9)
RECENT FULL: (2608, 53)

Annual trend latest years:


,year,fatalities,year_type,yoy_change,yoy_pct_change
23,2012,1300,Full year,23.0,1.801096
24,2013,1186,Full year,-114.0,-8.769231
25,2014,1150,Full year,-36.0,-3.035413
26,2015,1206,Full year,56.0,4.869565
27,2016,1294,Full year,88.0,7.296849
28,2017,1223,Full year,-71.0,-5.486862
29,2018,1135,Full year,-88.0,-7.195421
30,2019,1186,Full year,51.0,4.493392
31,2020,1097,Full year,-89.0,-7.504216
32,2021,1129,Full year,32.0,2.917046



Recent change 2024 to 2025:
2024: 1292
2025: 1316
Change: 24
Percent change: 1.86 %


,period,years,avg_annual_fatalities
0,Pre-COVID baseline,2017-2019,1181.333333
1,COVID period,2020-2021,1113.000000
2,Recent full years,2024-2025,1304.000000



Raw state ranking:


...


Top risk segments:


,state,remoteness_group,speed_band,road_user_group,fatalities,share_pct
138,VIC,Inner Regional,High speed 90-100,Vehicle occupant,112,4.126750
159,VIC,Major Cities,Urban arterial 60-80,Vulnerable road user,103,3.795136
73,QLD,Outer Regional,High speed 90-100,Vehicle occupant,101,3.721444
69,QLD,Major Cities,Urban arterial 60-80,Vulnerable road user,90,3.316139
24,NSW,Outer Regional,High speed 90-100,Vehicle occupant,89,3.279293
21,NSW,Major Cities,Urban arterial 60-80,Vulnerable road user,86,3.168755
8,NSW,Inner Regional,High speed 90-100,Vehicle occupant,81,2.984525
158,VIC,Major Cities,Urban arterial 60-80,Vehicle occupant,76,2.800295
20,NSW,Major Cities,Urban arterial 60-80,Vehicle occupant,73,2.689757
55,QLD,Inner Regional,High speed 90-100,Vehicle occupant,69,2.542373



Day x time heatmap data:


,dayweek,time_band,fatalities
7,Monday,Late night 00-05,50
8,Monday,Morning commute 06-08,39
5,Monday,Daytime 09-15,145
6,Monday,Evening commute 16-18,55
9,Monday,Night 19-23,61
27,Tuesday,Late night 00-05,31
28,Tuesday,Morning commute 06-08,46
25,Tuesday,Daytime 09-15,107
26,Tuesday,Evening commute 16-18,62
29,Tuesday,Night 19-23,67


EDA outputs saved to: /content/eda_outputs


In [9]:
# ============================================================
# STORY-LEVEL EDA: Final Tableau Narrative Tables
# ============================================================

import pandas as pd
import numpy as np
import os

OUT = "/content/story_outputs"
os.makedirs(OUT, exist_ok=True)

recent = master[master["analysis_window"].eq("Recent: 2024-Jan 2026")].copy()
recent_full = master[master["full_recent_year"].eq("2024-2025 full years")].copy()

# ------------------------------------------------------------
# 1. Create story archetypes
# ------------------------------------------------------------

def assign_archetype(row):
    if (
        row["remoteness_group"] in ["Inner Regional", "Outer Regional", "Remote / very remote"]
        and row["speed_band"] in ["High speed 90-100", "Very high speed 110+"]
        and row["road_user_group"] == "Vehicle occupant"
    ):
        return "Regional high-speed vehicle occupants"

    if (
        row["remoteness_group"] == "Major Cities"
        and row["speed_band"] in ["Urban arterial 60-80", "Local / urban <=50"]
        and row["road_user_group"] == "Vulnerable road user"
    ):
        return "Urban vulnerable road users"

    if (
        row["remoteness_group"] == "Major Cities"
        and row["speed_band"] == "Urban arterial 60-80"
        and row["road_user_group"] == "Vehicle occupant"
    ):
        return "Urban arterial vehicle occupants"

    return "Other / mixed pattern"

recent["story_archetype"] = recent.apply(assign_archetype, axis=1)

archetype_summary = (
    recent.groupby("story_archetype", as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

archetype_summary["share_pct"] = (
    archetype_summary["fatalities"] / archetype_summary["fatalities"].sum() * 100
)

display(archetype_summary)

# ------------------------------------------------------------
# 2. Archetype by state
# ------------------------------------------------------------

archetype_state = (
    recent.groupby(["state", "story_archetype"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

archetype_state["share_within_state_pct"] = (
    archetype_state["fatalities"]
    / archetype_state.groupby("state")["fatalities"].transform("sum")
    * 100
)

archetype_state = archetype_state.sort_values(
    ["state", "fatalities"], ascending=[True, False]
)

display(archetype_state)

# ------------------------------------------------------------
# 3. State change from 2024 to 2025
# ------------------------------------------------------------

state_year_change = (
    recent_full.groupby(["state", "year"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

state_change = state_year_change.pivot(
    index="state",
    columns="year",
    values="fatalities"
).reset_index()

state_change.columns.name = None
state_change["change_2024_to_2025"] = state_change[2025] - state_change[2024]
state_change["pct_change_2024_to_2025"] = (
    state_change["change_2024_to_2025"] / state_change[2024] * 100
)

state_change = state_change.sort_values("change_2024_to_2025", ascending=False)

display(state_change)

# ------------------------------------------------------------
# 4. Road environment summary
# ------------------------------------------------------------

environment_summary = (
    recent.groupby(["remoteness_group", "speed_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

environment_summary["share_pct"] = (
    environment_summary["fatalities"] / environment_summary["fatalities"].sum() * 100
)

display(environment_summary.head(20))

# ------------------------------------------------------------
# 5. Human exposure summary
# ------------------------------------------------------------

human_summary = (
    recent.groupby(["road_user_group", "road_user", "age_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

human_summary["share_pct"] = (
    human_summary["fatalities"] / human_summary["fatalities"].sum() * 100
)

display(human_summary.head(25))

# ------------------------------------------------------------
# 6. Timing summary
# ------------------------------------------------------------

day_time_story = (
    recent.groupby(["dayweek", "time_band"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

day_time_story["share_pct"] = (
    day_time_story["fatalities"] / day_time_story["fatalities"].sum() * 100
)

display(day_time_story.head(15))

# ------------------------------------------------------------
# 7. KPI table for Tableau cards
# ------------------------------------------------------------

kpi_summary = pd.DataFrame({
    "metric": [
        "Latest fatalities 2024-Jan 2026",
        "Full-year fatalities 2024",
        "Full-year fatalities 2025",
        "Change 2024 to 2025",
        "Percent change 2024 to 2025",
        "Highest population-adjusted risk state",
        "Highest population-adjusted risk value",
        "Top story archetype"
    ],
    "value": [
        recent["deaths"].sum(),
        recent_full[recent_full["year"].eq(2024)]["deaths"].sum(),
        recent_full[recent_full["year"].eq(2025)]["deaths"].sum(),
        recent_full[recent_full["year"].eq(2025)]["deaths"].sum()
        - recent_full[recent_full["year"].eq(2024)]["deaths"].sum(),
        round(
            (
                recent_full[recent_full["year"].eq(2025)]["deaths"].sum()
                - recent_full[recent_full["year"].eq(2024)]["deaths"].sum()
            )
            / recent_full[recent_full["year"].eq(2024)]["deaths"].sum()
            * 100,
            2
        ),
        state_risk.iloc[0]["state"],
        round(state_risk.iloc[0]["fatalities_per_100k"], 1),
        archetype_summary.iloc[0]["story_archetype"]
    ]
})

display(kpi_summary)

# ------------------------------------------------------------
# 8. Export story outputs
# ------------------------------------------------------------

archetype_summary.to_csv(f"{OUT}/story_archetype_summary.csv", index=False)
archetype_state.to_csv(f"{OUT}/story_archetype_by_state.csv", index=False)
state_change.to_csv(f"{OUT}/story_state_change_2024_2025.csv", index=False)
environment_summary.to_csv(f"{OUT}/story_environment_summary.csv", index=False)
human_summary.to_csv(f"{OUT}/story_human_exposure_summary.csv", index=False)
day_time_story.to_csv(f"{OUT}/story_day_time_summary.csv", index=False)
kpi_summary.to_csv(f"{OUT}/story_kpi_summary.csv", index=False)

print("Story outputs saved to:", OUT)


,story_archetype,fatalities,share_pct
0,Other / mixed pattern,1059,39.019897
1,Regional high-speed vehicle occupants,874,32.203390
3,Urban vulnerable road users,506,18.644068
2,Urban arterial vehicle occupants,275,10.132646


,state,story_archetype,fatalities,share_within_state_pct
0,ACT,Other / mixed pattern,8,34.782609
3,ACT,Urban vulnerable road users,8,34.782609
2,ACT,Urban arterial vehicle occupants,6,26.086957
1,ACT,Regional high-speed vehicle occupants,1,4.347826
4,NSW,Other / mixed pattern,250,35.161744
5,NSW,Regional high-speed vehicle occupants,245,34.458509
7,NSW,Urban vulnerable road users,143,20.112518
6,NSW,Urban arterial vehicle occupants,73,10.267229
8,NT,Other / mixed pattern,69,69.696970
9,NT,Regional high-speed vehicle occupants,30,30.303030


,state,2024,2025,change_2024_to_2025,pct_change_2024_to_2025
1,NSW,327,358,31,9.480122
5,TAS,31,44,13,41.935484
6,VIC,284,290,6,2.112676
3,QLD,302,307,5,1.655629
4,SA,89,87,-2,-2.247191
0,ACT,11,9,-2,-18.181818
7,WA,188,183,-5,-2.659574
2,NT,60,38,-22,-36.666667


,remoteness_group,speed_band,fatalities,share_pct
8,Major Cities,Urban arterial 60-80,640,23.581430
0,Inner Regional,High speed 90-100,379,13.964628
10,Outer Regional,High speed 90-100,311,11.459101
3,Inner Regional,Urban arterial 60-80,287,10.574797
6,Major Cities,Local / urban <=50,217,7.995578
13,Outer Regional,Urban arterial 60-80,128,4.716286
19,Remote / very remote,Very high speed 110+,106,3.905674
14,Outer Regional,Very high speed 110+,102,3.758290
4,Inner Regional,Very high speed 110+,97,3.574060
1,Inner Regional,Local / urban <=50,85,3.131909


,road_user_group,road_user,age_band,fatalities,share_pct
9,Vehicle occupant,Driver,40-64,385,14.185704
10,Vehicle occupant,Driver,65+,311,11.459101
8,Vehicle occupant,Driver,26-39,288,10.611643
7,Vehicle occupant,Driver,17-25,206,7.590273
25,Vulnerable road user,Motorcycle rider,40-64,190,7.000737
24,Vulnerable road user,Motorcycle rider,26-39,148,5.453206
37,Vulnerable road user,Pedestrian,65+,142,5.232130
23,Vulnerable road user,Motorcycle rider,17-25,135,4.974208
36,Vulnerable road user,Pedestrian,40-64,118,4.347826
13,Vehicle occupant,Passenger,17-25,101,3.721444


,dayweek,time_band,fatalities,share_pct
15,Sunday,Daytime 09-15,198,7.295505
20,Thursday,Daytime 09-15,161,5.932203
0,Friday,Daytime 09-15,158,5.821665
10,Saturday,Daytime 09-15,153,5.637436
5,Monday,Daytime 09-15,145,5.342668
30,Wednesday,Daytime 09-15,127,4.679440
25,Tuesday,Daytime 09-15,107,3.942520
4,Friday,Night 19-23,96,3.537214
14,Saturday,Night 19-23,91,3.352985
34,Wednesday,Night 19-23,84,3.095063


,metric,value
0,Latest fatalities 2024-Jan 2026,2714
1,Full-year fatalities 2024,1292
2,Full-year fatalities 2025,1316
3,Change 2024 to 2025,24
4,Percent change 2024 to 2025,1.86
5,Highest population-adjusted risk state,NT
6,Highest population-adjusted risk value,37.3
7,Top story archetype,Other / mixed pattern


Story outputs saved to: /content/story_outputs


In [10]:
# ============================================================
# FINAL STORY SEGMENTATION FOR TABLEAU
# Creates stronger HD dashboard fields
# ============================================================

import pandas as pd
import numpy as np
import os

OUT = "/content/final_tableau_story_outputs"
os.makedirs(OUT, exist_ok=True)

master_story = master.copy()

# ------------------------------------------------------------
# 1. Create intervention theatre
# ------------------------------------------------------------

regional_high_speed = (
    master_story["remoteness_group"].isin([
        "Inner Regional",
        "Outer Regional",
        "Remote / very remote"
    ])
    & master_story["speed_band"].isin([
        "High speed 90-100",
        "Very high speed 110+"
    ])
)

urban_streets = (
    master_story["remoteness_group"].eq("Major Cities")
    & master_story["speed_band"].isin([
        "Urban arterial 60-80",
        "Local / urban <=50"
    ])
)

master_story["intervention_theatre"] = np.select(
    [regional_high_speed, urban_streets],
    [
        "Regional high-speed roads",
        "Major-city urban streets"
    ],
    default="Other road contexts"
)

# ------------------------------------------------------------
# 2. Add policy target group
# ------------------------------------------------------------

master_story["policy_target_group"] = np.select(
    [
        master_story["intervention_theatre"].eq("Regional high-speed roads")
        & master_story["road_user_group"].eq("Vehicle occupant"),

        master_story["intervention_theatre"].eq("Major-city urban streets")
        & master_story["road_user_group"].eq("Vulnerable road user")
    ],
    [
        "Regional occupants",
        "Urban vulnerable users"
    ],
    default="Other / mixed users"
)

# ------------------------------------------------------------
# 3. Latest window summaries
# ------------------------------------------------------------

recent_story = master_story[
    master_story["analysis_window"].eq("Recent: 2024-Jan 2026")
].copy()

theatre_summary = (
    recent_story.groupby("intervention_theatre", as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

theatre_summary["share_pct"] = (
    theatre_summary["fatalities"] / theatre_summary["fatalities"].sum() * 100
)

display(theatre_summary)

target_group_summary = (
    recent_story.groupby("policy_target_group", as_index=False)
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

target_group_summary["share_pct"] = (
    target_group_summary["fatalities"] / target_group_summary["fatalities"].sum() * 100
)

display(target_group_summary)

# ------------------------------------------------------------
# 4. Theatre x road user group
# ------------------------------------------------------------

theatre_road_user = (
    recent_story.groupby(
        ["intervention_theatre", "road_user_group"],
        as_index=False
    )
    .agg(fatalities=("deaths", "sum"))
)

theatre_road_user["share_within_theatre_pct"] = (
    theatre_road_user["fatalities"]
    / theatre_road_user.groupby("intervention_theatre")["fatalities"].transform("sum")
    * 100
)

theatre_road_user = theatre_road_user.sort_values(
    ["intervention_theatre", "fatalities"],
    ascending=[True, False]
)

display(theatre_road_user)

# ------------------------------------------------------------
# 5. Theatre by state
# ------------------------------------------------------------

theatre_state = (
    recent_story.groupby(["state", "intervention_theatre"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

theatre_state["share_within_state_pct"] = (
    theatre_state["fatalities"]
    / theatre_state.groupby("state")["fatalities"].transform("sum")
    * 100
)

theatre_state = theatre_state.sort_values(
    ["state", "fatalities"],
    ascending=[True, False]
)

display(theatre_state)

# ------------------------------------------------------------
# 6. Theatre by time
# ------------------------------------------------------------

theatre_time = (
    recent_story.groupby(
        ["intervention_theatre", "dayweek", "time_band"],
        as_index=False
    )
    .agg(fatalities=("deaths", "sum"))
    .sort_values("fatalities", ascending=False)
)

display(theatre_time.head(25))

# ------------------------------------------------------------
# 7. Final KPI table
# ------------------------------------------------------------

recent_full_story = master_story[
    master_story["full_recent_year"].eq("2024-2025 full years")
].copy()

total_latest = recent_story["deaths"].sum()
fatalities_2024 = recent_full_story[recent_full_story["year"].eq(2024)]["deaths"].sum()
fatalities_2025 = recent_full_story[recent_full_story["year"].eq(2025)]["deaths"].sum()

priority_share = theatre_summary[
    theatre_summary["intervention_theatre"].isin([
        "Regional high-speed roads",
        "Major-city urban streets"
    ])
]["fatalities"].sum() / total_latest * 100

kpi_final = pd.DataFrame({
    "metric": [
        "Latest fatalities 2024-Jan 2026",
        "Full-year fatalities 2025",
        "Change from 2024 to 2025",
        "Highest population-adjusted risk state",
        "Highest population-adjusted risk per 100k",
        "Share in two priority theatres"
    ],
    "value": [
        total_latest,
        fatalities_2025,
        fatalities_2025 - fatalities_2024,
        state_risk.iloc[0]["state"],
        round(state_risk.iloc[0]["fatalities_per_100k"], 1),
        round(priority_share, 1)
    ]
})

display(kpi_final)

# ------------------------------------------------------------
# 8. Export final Tableau-ready files
# ------------------------------------------------------------

master_story.to_csv(f"{OUT}/master_dashboard_story.csv", index=False)
theatre_summary.to_csv(f"{OUT}/theatre_summary_latest.csv", index=False)
target_group_summary.to_csv(f"{OUT}/target_group_summary_latest.csv", index=False)
theatre_road_user.to_csv(f"{OUT}/theatre_road_user_latest.csv", index=False)
theatre_state.to_csv(f"{OUT}/theatre_state_latest.csv", index=False)
theatre_time.to_csv(f"{OUT}/theatre_time_latest.csv", index=False)
kpi_final.to_csv(f"{OUT}/kpi_final.csv", index=False)

print("Final Tableau story outputs saved to:", OUT)


,intervention_theatre,fatalities,share_pct
2,Regional high-speed roads,1065,39.240973
0,Major-city urban streets,857,31.577008
1,Other road contexts,792,29.182019


,policy_target_group,fatalities,share_pct
0,Other / mixed users,1334,49.152542
1,Regional occupants,874,32.203390
2,Urban vulnerable users,506,18.644068


,intervention_theatre,road_user_group,fatalities,share_within_theatre_pct
2,Major-city urban streets,Vulnerable road user,506,59.043174
1,Major-city urban streets,Vehicle occupant,328,38.273046
0,Major-city urban streets,Unknown,23,2.683781
4,Other road contexts,Vehicle occupant,417,52.651515
5,Other road contexts,Vulnerable road user,358,45.202020
3,Other road contexts,Unknown,17,2.146465
7,Regional high-speed roads,Vehicle occupant,874,82.065728
8,Regional high-speed roads,Vulnerable road user,187,17.558685
6,Regional high-speed roads,Unknown,4,0.375587


,state,intervention_theatre,fatalities,share_within_state_pct
0,ACT,Major-city urban streets,14,60.869565
1,ACT,Other road contexts,6,26.086957
2,ACT,Regional high-speed roads,3,13.043478
5,NSW,Regional high-speed roads,282,39.662447
3,NSW,Major-city urban streets,237,33.333333
4,NSW,Other road contexts,192,27.004219
6,NT,Other road contexts,63,63.636364
7,NT,Regional high-speed roads,36,36.363636
10,QLD,Regional high-speed roads,245,38.827258
8,QLD,Major-city urban streets,202,32.012678


,intervention_theatre,dayweek,time_band,fatalities
85,Regional high-speed roads,Sunday,Daytime 09-15,96
70,Regional high-speed roads,Friday,Daytime 09-15,72
80,Regional high-speed roads,Saturday,Daytime 09-15,71
90,Regional high-speed roads,Thursday,Daytime 09-15,63
50,Other road contexts,Sunday,Daytime 09-15,60
75,Regional high-speed roads,Monday,Daytime 09-15,58
100,Regional high-speed roads,Wednesday,Daytime 09-15,51
20,Major-city urban streets,Thursday,Daytime 09-15,51
35,Other road contexts,Friday,Daytime 09-15,50
45,Other road contexts,Saturday,Daytime 09-15,49


,metric,value
0,Latest fatalities 2024-Jan 2026,2714
1,Full-year fatalities 2025,1316
2,Change from 2024 to 2025,24
3,Highest population-adjusted risk state,NT
4,Highest population-adjusted risk per 100k,37.3
5,Share in two priority theatres,70.8


Final Tableau story outputs saved to: /content/final_tableau_story_outputs


In [11]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/final_tableau_story_outputs",
    "zip",
    "/content/final_tableau_story_outputs"
)

files.download("/content/final_tableau_story_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>